<a href="https://www.kaggle.com/code/avikdas567/geohab-2026-mlwg-competition?scriptVersionId=316293476" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# GeoHab 2026 MLWG Competition  
## Predicting Benthic Habitat Classes from Multibeam Sonar Data

This notebook presents a complete geospatial machine learning workflow for the GeoHab 2026 MLWG Competition, focused on predicting benthic habitat classes from multibeam echosounder (MBES) data collected at Refuge Cove, Victoria, Australia.

The approach integrates:

- raster-based environmental feature extraction
- spatial feature engineering
- terrain and texture derivatives
- ensemble-based multiclass classification
- repeated stratified cross-validation
- weighted F1 evaluation

---

## Methodology

The workflow uses the provided:

- bathymetry raster
- backscatter raster
- training observations
- spatial coordinates

to derive predictive variables describing seabed structure and acoustic response characteristics.

Feature engineering includes:

- bathymetric gradients
- local spatial variability
- smoothed terrain representations
- acoustic texture measures
- coordinate interaction features

These predictors are subsequently used within an ensemble framework combining multiple gradient boosting approaches for multiclass habitat prediction.

---

## Modeling Strategy

The modeling pipeline includes:

- LightGBM
- CatBoost
- XGBoost

with probability-level ensembling across repeated stratified folds.

Model performance is evaluated using the competition metric:

### Weighted F1 Score

which accounts for class imbalance while preserving multiclass predictive quality.

---

## Study Context

The dataset originates from:

> Ierodiaconou, D., Schimel, A.C.G., Kennedy, D., Monk, J., Gaylard, G., Young, M., Diesing, M., Rattray, A., 2018. Combining pixel and object based image analysis of ultra-high resolution multibeam bathymetry and backscatter for habitat mapping in shallow marine waters. *Marine Geophysical Research*, 39, 271–288.

Target habitat classes include:

- ALG — Macroalgae Dominated Reef
- FMAT — Filamentous Mat
- NVB — No Visible Biota
- SGAM — Seagrass (*Amphibolis antarctica*)
- SGZ — Seagrass (*Zostera* sp.)

---

## Notes

The notebook is structured to support reproducibility and further experimentation with:

- alternative terrain descriptors
- spatially-aware validation strategies
- raster neighborhood statistics
- feature selection
- advanced ensemble weighting
- deep learning approaches using raster patches

---

In [1]:
import os
import gc
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from scipy.ndimage import gaussian_filter
from scipy.ndimage import sobel
from scipy.ndimage import generic_filter
from scipy.ndimage import uniform_filter

import rasterio

from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import f1_score
from sklearn.preprocessing import LabelEncoder

from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from xgboost import XGBClassifier

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

DATA_PATH = "/kaggle/input/competitions/geohab-mlwg-competition-2026"

TRAIN_PATH = f"{DATA_PATH}/train.csv"
TEST_PATH = f"{DATA_PATH}/test.csv"
SAMPLE_PATH = f"{DATA_PATH}/sample_submission.csv"

BATHY_PATH = f"{DATA_PATH}/MBES/bathymetry.tif"
BACK_PATH = f"{DATA_PATH}/MBES/backscatter.tif"

print("Files ready")

Files ready


In [2]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

print(train.shape, test.shape)
display(train.head())

TARGET = "class"
ID_COL = "ID"

features_base = ["x", "y"]

(6256, 3) (98, 3)


,class,x,y
0,NVB,453594.477237,5.679192e+06
1,FMAT,453561.906453,5.679109e+06
2,ALG,453744.452238,5.679033e+06
3,ALG,453863.445302,5.679038e+06
4,ALG,453964.611906,5.679017e+06


In [3]:
def fast_std(a, size):
    mean = uniform_filter(a, size=size)
    mean_sq = uniform_filter(a*a, size=size)
    return np.sqrt(np.maximum(mean_sq - mean**2, 0))

def build_raster_features(raster_path, prefix):

    src = rasterio.open(raster_path)

    arr = src.read(1).astype(np.float32)

    arr = np.nan_to_num(arr, nan=np.nanmedian(arr))

    gx = sobel(arr, axis=0)
    gy = sobel(arr, axis=1)

    slope = np.sqrt(gx**2 + gy**2)

    smooth3 = gaussian_filter(arr, sigma=3)
    smooth7 = gaussian_filter(arr, sigma=7)

    local3 = fast_std(arr, 3)
    local7 = fast_std(arr, 7)

    transforms = {
        f"{prefix}_raw": arr,
        f"{prefix}_slope": slope,
        f"{prefix}_smooth3": smooth3,
        f"{prefix}_smooth7": smooth7,
        f"{prefix}_std3": local3,
        f"{prefix}_std7": local7,
    }

    return src, transforms

bath_src, bath_feats = build_raster_features(BATHY_PATH, "bath")
back_src, back_feats = build_raster_features(BACK_PATH, "back")

print("Raster features created")

Raster features created


In [4]:
def sample_features(df):

    coords = list(zip(df["x"].values, df["y"].values))

    out = pd.DataFrame(index=df.index)

    for name, arr in bath_feats.items():
        vals = []
        for x, y in coords:
            row, col = bath_src.index(x, y)

            row = np.clip(row, 0, arr.shape[0]-1)
            col = np.clip(col, 0, arr.shape[1]-1)

            vals.append(arr[row, col])

        out[name] = vals

    for name, arr in back_feats.items():
        vals = []
        for x, y in coords:
            row, col = back_src.index(x, y)

            row = np.clip(row, 0, arr.shape[0]-1)
            col = np.clip(col, 0, arr.shape[1]-1)

            vals.append(arr[row, col])

        out[name] = vals

    out["x"] = df["x"].values
    out["y"] = df["y"].values

    out["xy"] = out["x"] * out["y"]
    out["x2"] = out["x"] ** 2
    out["y2"] = out["y"] ** 2

    out["bath_back_ratio"] = out["bath_raw"] / (np.abs(out["back_raw"]) + 1e-3)
    out["bath_back_diff"] = out["bath_raw"] - out["back_raw"]

    return out

X_train = sample_features(train)
X_test = sample_features(test)

print(X_train.shape, X_test.shape)

display(X_train.head())

(6256, 19) (98, 19)


,bath_raw,bath_slope,bath_smooth3,bath_smooth7,bath_std3,bath_std7,back_raw,back_slope,back_smooth3,back_smooth7,back_std3,back_std7,x,y,xy,x2,y2,bath_back_ratio,bath_back_diff
0,-2.822883,0.490155,-771.701782,-2795.651367,0.055604,0.051582,-18.669088,1.011403,-785.917419,-2806.622314,0.298106,0.684096,453594.477237,5.679192e+06,2.576050e+12,2.057479e+11,3.225323e+13,-0.151198,15.846205
1,-5.668992,0.056516,-5.699245,-5.743821,0.021749,0.047921,-20.558794,5.379626,-19.881098,-20.116619,0.565584,0.958409,453561.906453,5.679109e+06,2.575827e+12,2.057184e+11,3.225227e+13,-0.275732,14.889803
2,-9.288988,1.095672,-9.358536,-9.335884,0.114620,0.123155,-23.389690,0.877005,-22.825531,-22.644981,0.154087,1.029680,453744.452238,5.679033e+06,2.576830e+12,2.058840e+11,3.225142e+13,-0.397123,14.100702
3,-7.263793,0.912235,-7.229817,-7.230651,0.111157,0.141840,-21.189917,9.537559,-22.130901,-22.365448,1.098733,1.255238,453863.445302,5.679038e+06,2.577508e+12,2.059920e+11,3.225147e+13,-0.342779,13.926123
4,-9.784807,2.092018,-9.656490,-9.548962,0.240591,0.356530,-22.449720,9.959623,-24.426985,-25.935165,1.050048,1.351269,453964.611906,5.679017e+06,2.578073e+12,2.060839e+11,3.225123e+13,-0.435835,12.664913


In [5]:
le = LabelEncoder()

y = le.fit_transform(train[TARGET])

classes = le.classes_

print(classes)

oof = np.zeros((len(train), len(classes)), dtype=np.float32)
preds = np.zeros((len(test), len(classes)), dtype=np.float32)

rskf = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=2,
    random_state=SEED
)

scores = []

for fold, (tr_idx, va_idx) in enumerate(rskf.split(X_train, y)):

    print(f"\n========== Fold {fold+1} ==========")

    X_tr = X_train.iloc[tr_idx]
    y_tr = y[tr_idx]

    X_va = X_train.iloc[va_idx]
    y_va = y[va_idx]

    # =========================
    # LIGHTGBM
    # =========================
    lgb_model = LGBMClassifier(
        n_estimators=500,
        learning_rate=0.03,
        max_depth=-1,
        num_leaves=63,
        subsample=0.85,
        colsample_bytree=0.85,
        objective="multiclass",
        random_state=SEED + fold,
        device="gpu",
        verbosity=-1
    )

    # =========================
    # CATBOOST
    # =========================
    cat_model = CatBoostClassifier(
        iterations=500,
        learning_rate=0.03,
        depth=8,
        loss_function="MultiClass",
        eval_metric="TotalF1",
        random_seed=SEED + fold,
        verbose=False,
        task_type="GPU"
    )

    # =========================
    # XGBOOST
    # =========================
    xgb_model = XGBClassifier(
        n_estimators=500,
        learning_rate=0.03,
        max_depth=8,
        subsample=0.85,
        colsample_bytree=0.85,
        objective="multi:softprob",
        tree_method="hist",
        device="cuda",
        random_state=SEED + fold,
        eval_metric="mlogloss"
    )

    # =========================
    # TRAIN
    # =========================
    lgb_model.fit(X_tr, y_tr)
    cat_model.fit(X_tr, y_tr)
    xgb_model.fit(X_tr, y_tr)

    # =========================
    # VALIDATION PREDICTIONS
    # =========================
    p1 = lgb_model.predict_proba(X_va)
    p2 = cat_model.predict_proba(X_va)
    p3 = xgb_model.predict_proba(X_va)

    val_pred = (
        0.35 * p1 +
        0.35 * p2 +
        0.30 * p3
    )

    oof[va_idx] += val_pred

    fold_score = f1_score(
        y_va,
        np.argmax(val_pred, axis=1),
        average="weighted"
    )

    print("Weighted F1:", round(fold_score, 6))

    scores.append(fold_score)

    # =========================
    # TEST PREDICTIONS
    # =========================
    tp1 = lgb_model.predict_proba(X_test)
    tp2 = cat_model.predict_proba(X_test)
    tp3 = xgb_model.predict_proba(X_test)

    preds += (
        0.35 * tp1 +
        0.35 * tp2 +
        0.30 * tp3
    ) / (5 * 2)

    # =========================
    # CLEANUP
    # =========================
    del lgb_model
    del cat_model
    del xgb_model

    gc.collect()

print("\n==============================")
print("FINAL CV Weighted F1:", round(np.mean(scores), 6))
print("==============================")

['ALG' 'FMAT' 'NVB' 'SGAM' 'SGZ']

========== Fold 1 ==========


1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


Weighted F1: 0.988812

========== Fold 2 ==========
Weighted F1: 0.989605

========== Fold 3 ==========
Weighted F1: 0.986412

========== Fold 4 ==========
Weighted F1: 0.981503

========== Fold 5 ==========
Weighted F1: 0.985526

========== Fold 6 ==========
Weighted F1: 0.986367

========== Fold 7 ==========
Weighted F1: 0.984774

========== Fold 8 ==========
Weighted F1: 0.990415

========== Fold 9 ==========
Weighted F1: 0.988802

========== Fold 10 ==========
Weighted F1: 0.984783

FINAL CV Weighted F1: 0.9867


In [6]:
test_pred = np.argmax(preds, axis=1)

test_labels = le.inverse_transform(test_pred)

submission = pd.DataFrame({
    "ID": test["ID"],
    "class": test_labels
})

submission.to_csv("submission.csv", index=False)

display(submission)
print("'submission.csv' created successfully!")

,ID,class
0,1,SGZ
1,2,ALG
2,3,NVB
3,4,NVB
4,5,SGZ
...,...,...
93,94,NVB
94,95,NVB
95,96,SGZ
96,97,SGZ


'submission.csv' created successfully!
